# Week 7 Project: Document Question Answering System (RAG)

**Retrieval-Augmented Generation (RAG)** combines three ideas:
1. **Retrieval** — find the most relevant chunks from your document using vector similarity
2. **Augmentation** — inject those chunks as context into the LLM prompt
3. **Generation** — LLM produces a grounded, factual answer

```
PDF ──► Extract Text ──► Word-based Chunks ──► TF-IDF Vectors ──► Vector Store
                                                                        │
User Query ──────────────────────────────────────────────────► Cosine Similarity
                                                                        │
                                                                  Top-K Chunks
                                                                        │
                                                              Groq LLM (Llama 3.1)
                                                                        │
                                                                     Answer
```
> **Free API:** [Groq](https://console.groq.com) — sign up free, no credit card.

## Step 0: Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install pypdf groq --quiet
print("Dependencies ready!")

Dependencies ready!


## Step 1: Imports & API Key

In [2]:
import os, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from groq import Groq

os.environ["GROQ_API_KEY"] = "your_groq_api_key_here"
client = Groq()
print("All imports OK!")

All imports OK!


## Step 2: Document Ingestion — Load PDF

In [3]:
DOC_PATH = "/Users/gaurichopra/Desktop/3RD YEAR/LLM/LLM.M3/RLHF.pdf"

def load_pdf(path):
    """Extract text from every page of a PDF."""
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            pages.append(text)
    return "\n".join(pages), len(reader.pages)

raw_text, total_pages = load_pdf(DOC_PATH)
print(f"Document : {DOC_PATH.split('/')[-1]}")
print(f"Pages    : {total_pages}")
print(f"Characters: {len(raw_text):,}")
print(f"Words    : {len(raw_text.split()):,}")
print("\n--- Preview (first 500 chars) ---")
print(raw_text[:500])

Document : RLHF.pdf
Pages    : 32
Characters: 12,710
Words    : 1,745

--- Preview (first 500 chars) ---
Reinforcement Learning from Human Feedback (RLHF)
What is RLHF?
RLHF uses human preferences as the ultimate reward signal to fine-tune a Large
Language Model (LLM).
Why do LLMs need RLHF?
Even after pre-training and supervised fine-tuning (SFT), LLMs can still produce
responses that are subtly misaligned with human values or expectations


## Step 3: Text Chunking

We use **word-based chunking with overlap** (same strategy as production RAG systems).  
- `chunk_size = 150 words` — enough context per chunk  
- `overlap = 30 words` — avoids cutting answers at chunk boundaries

In [4]:
def chunk_text(text, chunk_size=150, overlap=30):
    """Split text into overlapping word-based chunks."""
    # Clean whitespace
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]
        chunks.append(" ".join(chunk_words))
        start += chunk_size - overlap   # slide forward with overlap
    return chunks

chunks = chunk_text(raw_text, chunk_size=150, overlap=30)

print(f"Total chunks  : {len(chunks)}")
print(f"Words/chunk   : ~150 with 30-word overlap")
print(f"\n--- Chunk 1 ---\n{chunks[0]}")
print(f"\n--- Chunk 2 ---\n{chunks[1]}")

Total chunks  : 15
Words/chunk   : ~150 with 30-word overlap

--- Chunk 1 ---
Reinforcement Learning from Human Feedback (RLHF) What is RLHF? RLHF uses human preferences as the ultimate reward signal to fine-tune a Large Language Model (LLM). Why do LLMs need RLHF? Even after pre-training and supervised fine-tuning (SFT), LLMs can still produce responses that are subtly misaligned with human values or expectations

--- Chunk 2 ---
responses that are subtly misaligned with human values or expectations or toxic language present in its training data. Lack of Safety Awareness: It might provide instructions for dangerous activities. Sycophancy: It might tell users what they want to hear rather than the truth. RLHF was a key technique behind the impressive capabilities of models like ChatGPT.


## Step 4: Embedding & Vector Store (TF-IDF)

Each chunk is converted to a **TF-IDF vector** — a sparse representation where each dimension corresponds to a vocabulary term, weighted by how important that term is in the chunk vs the whole corpus.  
No model download required.

In [5]:
# Build TF-IDF vector store
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),      # unigrams + bigrams for better matching
    min_df=1,
    max_df=0.95
)
chunk_vectors = vectorizer.fit_transform(chunks)

print(f"Vector store built!")
print(f"  Chunks : {chunk_vectors.shape[0]}")
print(f"  Terms  : {chunk_vectors.shape[1]:,} (unigrams + bigrams)")
print(f"  Sparsity: {(1 - chunk_vectors.nnz / (chunk_vectors.shape[0] * chunk_vectors.shape[1])):.1%}")

Vector store built!
  Chunks : 15
  Terms  : 1,739 (unigrams + bigrams)
  Sparsity: 88.7%


## Step 5: Retrieval — Query Processing & Similarity Search

In [6]:
def retrieve(query, top_k=3):
    """Embed query and retrieve top-K most similar chunks via cosine similarity."""
    q_vec = vectorizer.transform([query])
    scores = cosine_similarity(q_vec, chunk_vectors).flatten()
    top_indices = scores.argsort()[::-1][:top_k]
    return [
        {"chunk_id": int(i), "text": chunks[i], "score": float(scores[i])}
        for i in top_indices
        if scores[i] > 0      # only return chunks with actual overlap
    ]

# Test retrieval
print("=" * 65)
print("RETRIEVAL TEST: 'What is RLHF?'")
print("=" * 65)
for r in retrieve("What is RLHF?"):
    print(f"\n[Chunk {r['chunk_id']} | Score: {r['score']:.4f}]")
    print(r["text"])
    print("-" * 40)

RETRIEVAL TEST: 'What is RLHF?'

[Chunk 2 | Score: 0.1697]
responses that are subtly misaligned with human values or expectations or toxic language present in its training data. Lack of Safety Awareness: It might provide instructions for dangerous activities. Sycophancy: It might tell users what they want to hear rather than the truth. RLHF was a key technique behind the impressive capabilities of models like ChatGPT.
----------------------------------------

[Chunk 13 | Score: 0.0979]
as the generation of phrases or words that inflate alignment metrics but compromise overall quality and accuracy. Limitations of RLHF The initial human effort to create a robust reward model is substantial. Large teams of labellers, sometimes numbering in the thousands, are required to evaluate numerous interactions.
----------------------------------------

[Chunk 6 | Score: 0.0809]
pair that accurately reflects the learned human preferences. Who/What is the reward model? The reward model is an LLM with

## Step 6: Answer Generation with Groq (Free — Llama 3.1 8B)

In [7]:
def ask(query, top_k=3):
    """Full RAG pipeline: retrieve → augment → generate."""
    
    # 1. RETRIEVE — find relevant chunks
    sources = retrieve(query, top_k)
    if not sources:
        return "No relevant content found in the document for this question.", []
    
    # 2. AUGMENT — build context string
    context = "\n\n".join(
        f"[Source {i+1} | Chunk #{s['chunk_id']}]\n{s['text']}"
        for i, s in enumerate(sources)
    )
    
    # 3. GENERATE — call Groq LLM with context-grounded prompt
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a precise Q&A assistant. "
                    "Answer the user's question using ONLY the provided context. "
                    "Be concise and factual. "
                    "If the answer is not in the context, say: 'This information is not covered in the document.'"
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {query}"
            }
        ],
        max_tokens=350,
        temperature=0
    )
    
    return response.choices[0].message.content.strip(), sources


def ask_and_display(query):
    print(f"\nQ: {query}")
    print("=" * 65)
    answer, sources = ask(query)
    print(f"A: {answer}")
    if sources:
        print(f"\nRetrieved {len(sources)} source(s):")
        for s in sources:
            print(f"  [Chunk #{s['chunk_id']} | Score: {s['score']:.3f}] {s['text'][:90]}...")
    print("=" * 65)

print("Pipeline ready! Functions defined.")

Pipeline ready! Functions defined.


## Step 7: Ask Questions About the RLHF Document

In [8]:
ask_and_display("What is RLHF?")


Q: What is RLHF?
A: Reinforcement Learning from Human Feedback (RLHF) uses human preferences as the ultimate reward signal to fine-tune a Large Language Model (LLM).

Retrieved 3 source(s):
  [Chunk #2 | Score: 0.170] responses that are subtly misaligned with human values or expectations or toxic language...
  [Chunk #13 | Score: 0.098] as the generation of phrases or words that inflate alignment metrics but compromise overa...
  [Chunk #6 | Score: 0.081] pair that accurately reflects the learned human preferences. Who/What is the reward model...


In [9]:
ask_and_display("How does reward modeling work in RLHF?")


Q: How does reward modeling work in RLHF?
A: Reward modeling in RLHF works by training a separate model, called the Reward Model (RM), to predict human preferences. The RM is trained on ranked data from human labellers, where they rank responses from best to worst. The ranked data is broken down into pairwise comparisons, and the RM is trained to predict the winner in each pair. Over time, it learns to assign a single, scalar "reward" score to any given prompt-response pair that accurately reflects the learned human preferences.

Retrieved 3 source(s):
  [Chunk #2 | Score: 0.167] responses that are subtly misaligned with human values or expectations or toxic language...
  [Chunk #6 | Score: 0.154] pair that accurately reflects the learned human preferences. Who/What is the reward model...
  [Chunk #5 | Score: 0.134] data for the Reward Model is gathered through a simple but powerful process: ranked data f...


In [10]:
ask_and_display("What is PPO and why is it used in RLHF?")


Q: What is PPO and why is it used in RLHF?
A: PPO (Proximal Policy Optimization) is a reinforcement learning algorithm used in RLHF due to its stability and ability to make smaller, more reliable updates, preventing catastrophic forgetting and ensuring the model improves its helpfulness without losing its core language skills.

Retrieved 3 source(s):
  [Chunk #8 | Score: 0.183] input can have multiple correct outputs (Write a poem?) Staying close to ref can preserve...
  [Chunk #9 | Score: 0.091] highly unstable. They might update the model so aggressively that it forgets how to produc...
  [Chunk #12 | Score: 0.080] trust region, ensuring that the old and new policies remain close to each other during upd...


In [11]:
ask_and_display("What are the main steps in the RLHF training pipeline?")


Q: What are the main steps in the RLHF training pipeline?
A: The main steps in the RLHF training pipeline are:

1. Phase 1: Supervised Fine-Tuning (SFT) — fine-tune the base LLM on a small, high-quality dataset.
2. Phase 2: Reward Model Training — train a Reward Model (RM) using ranked human feedback to predict human preferences.
3. Phase 3: RL Fine-Tuning — use the RM as a proxy for human feedback to further fine-tune the LLM using reinforcement learning (PPO).

Retrieved 3 source(s):
  [Chunk #6 | Score: 0.095] pair that accurately reflects the learned human preferences. Who/What is the reward model...
  [Chunk #2 | Score: 0.088] responses that are subtly misaligned with human values or expectations or toxic language...
  [Chunk #4 | Score: 0.071] is not a single step but a multi-stage process that builds upon itself. Phase 1: Supervise...


In [12]:
ask_and_display("What are the limitations and challenges of RLHF?")


Q: What are the limitations and challenges of RLHF?
A: The limitations and challenges of RLHF include:

1. Substantial initial human effort to create a robust reward model.
2. Large teams of labellers (sometimes numbering in the thousands) are required to evaluate numerous interactions.
3. Human effort becomes a limiting factor as the number of models and use cases increases.

Retrieved 3 source(s):
  [Chunk #14 | Score: 0.104] the alignment process. Reward hacking: the model finds ways to exploit the reward model...
  [Chunk #2 | Score: 0.096] responses that are subtly misaligned with human values or expectations or toxic language...
  [Chunk #13 | Score: 0.095] as the generation of phrases or words that inflate alignment metrics but compromise overa...


In [13]:
ask_and_display("What is the difference between RLHF and supervised fine-tuning?")


Q: What is the difference between RLHF and supervised fine-tuning?
A: RLHF (Reinforcement Learning with Human Feedback) and Supervised Fine-Tuning (SFT) differ in that SFT is the first phase of RLHF, where the base LLM is fine-tuned on a small, high-quality dataset to learn to follow instructions. RLHF then builds on SFT by using a Reward Model trained on human preferences and reinforcement learning (PPO) to further align the model — a multi-stage process that SFT alone cannot achieve.

Retrieved 3 source(s):
  [Chunk #4 | Score: 0.239] is not a single step but a multi-stage process that builds upon itself. Phase 1: Supervise...
  [Chunk #3 | Score: 0.113] was a key technique behind the impressive capabilities of models like ChatGPT. Applying RL...
  [Chunk #8 | Score: 0.095] input can have multiple correct outputs (Write a poem?) Staying close to ref can preserve...


In [14]:
ask_and_display("What role do human labelers play in RLHF?")


Q: What role do human labelers play in RLHF?
A: Human labelers provide initial feedback to create a robust reward model by ranking responses from best to worst. This ranked data is used to train the Reward Model (RM), which then acts as a fast and scalable proxy for human feedback to fine-tune the LLM.

Retrieved 3 source(s):
  [Chunk #6 | Score: 0.118] pair that accurately reflects the learned human preferences. Who/What is the reward model...
  [Chunk #13 | Score: 0.114] as the generation of phrases or words that inflate alignment metrics but compromise overa...
  [Chunk #2 | Score: 0.099] responses that are subtly misaligned with human values or expectations or toxic language...


## Step 8: Interactive Q&A Loop

In [ ]:
print("Interactive RAG Q&A — type 'quit' to exit\n")
while True:
    user_input = input("Your question: ").strip()
    if user_input.lower() in ("quit", "exit", "q"):
        print("Done!")
        break
    if user_input:
        ask_and_display(user_input)

## Summary

| Step | Component | Tool / Method |
|------|-----------|---------------|
| 1. Ingestion | Load PDF pages into raw text | `pypdf` |
| 2. Chunking | 150-word chunks with 30-word overlap | Custom function |
| 3. Embedding | TF-IDF with unigrams + bigrams | `sklearn` |
| 4. Vector Store | Sparse TF-IDF matrix | `sklearn` |
| 5. Retrieval | Cosine similarity → top-K chunks | `sklearn` |
| 6. Augmentation | Retrieved chunks injected as context | Prompt engineering |
| 7. Generation | Grounded answer | **Groq — Llama 3.1 8B (FREE)** |

## Possible Improvements
- Use `sentence-transformers` (dense embeddings) instead of TF-IDF for semantic search
- Add FAISS vector index for faster search at scale  
- Implement **hybrid search** (TF-IDF keyword + semantic vector)
- Add **re-ranking** with a cross-encoder to improve top-K precision
- Add **conversation history** for multi-turn Q&A